####  Tensor

In [2]:
import torch

# ========================= 原生torch.Tensor核心原理+实操演示 =========================
# PyTorch底层双层架构（后续手写迷你框架会完整复刻这套结构）
# 1. StorageImpl：连续一维内存块，存放原始数值，支持CPU/GPU设备切换
# 2. TensorImpl：张量视图结构体，维护shape形状、stride步幅、offset偏移、梯度标记grad_fn/requires_grad/grad
# 原生list三大短板对照Tensor优势：
# ① list数据零散分布，无法整块迁移GPU；Tensor依托连续Storage内存，可一键设备迁移
# ② list无梯度追踪组件；Tensor自带Autograd自动求导链路，运算生成算子Node节点，grad_fn绑定生成算子
# ③ list没有视图复用机制；Tensor通过修改视图参数实现reshape/切片，不复制底层Storage数据

token_ids = [1, 5, 23, 432, 333]
token_tensor = torch.tensor(token_ids)

# 演示1：设备迁移（list无法直接调用cuda接口）
# token_ids.cuda()  # 原生列表直接报错
# token_gpu = token_tensor.to("cuda")

# 演示2：自动微分（底层生成Pow算子Node，grad_fn指向算子实例）
# 链式推导：y=x²，dy/dx=2x；x=2.0代入得梯度=4
# 前向链路：x(叶子张量) → Pow算子 → y(中间输出张量)
# 反向链路：初始梯度dL/dy=1 → 算子计算dL/dx=1*2x=4，写入x.grad
x = torch.tensor([2.0], requires_grad=True)
y = x ** 2
print("原生算子grad_fn：", y.grad_fn)
y.backward()
print(f"分步推导：导数式2x、x=2代入得4，x梯度：{x.grad}")

# 演示3：视图共享内存 VS clone独立拷贝（视图仅改TensorImpl、复用Storage；clone新建独立Storage）
data = torch.tensor([1,2,3,4])
view_tensor = data.reshape(2, 2)
data[0] = 999
print("视图共享内存结果：\n", view_tensor)

clone_tensor = data.clone()
clone_tensor[0] = 0
print("clone独立内存结果：", data, clone_tensor)

# 演示4：常数张量特性（同样双层结构、requires_grad=False，参与运算不记录梯度）
const_2 = torch.tensor([2.0])
z = const_2 * x
z.backward()
print("常数参与运算叠加后x梯度：", x.grad)

# 小结
# Tensor = StorageImpl(数据存储) + TensorImpl(视图+梯度控制)
# 叶子张量保存梯度、中间张量中转梯度；加法梯度原样透传，乘法梯度用另一侧输入缩放
print("================ Tensor模块结束 ================\n")


原生算子grad_fn： <PowBackward0 object at 0x7f867434ae60>
分步推导：导数式2x、x=2代入得4，x梯度：tensor([4.])
视图共享内存结果：
 tensor([[999,   2],
        [  3,   4]])
clone独立内存结果： tensor([999,   2,   3,   4]) tensor([0, 2, 3, 4])
常数参与运算叠加后x梯度： tensor([6.])
================ Tensor模块结束 ================



#### nn.Parameter

In [1]:
import torch
import torch.nn as nn

"""
前置知识承接：
nn.Parameter 是 torch.Tensor 的子类，底层沿用 Tensor 双层存储结构（StorageImpl存数值、TensorImpl管视图梯度），二者加减乘除、链式求导这类基础运算、梯度计算规则完全一致。
## 普通Tensor vs nn.Parameter 核心区别+实例对照
1. 参数自动收录差异（最核心区别）
普通Tensor即便手动开启requires_grad=True、绑定成模型类成员变量，nn.Module也不会自动识别收录；调用model.parameters()遍历、初始化优化器时读取不到这个张量，优化器执行step无法自动更新它的数值，想要修改只能手动编写代码逐个赋值更新。
nn.Parameter只要挂载为模型成员变量，父类nn.Module初始化阶段会自动扫描登记进专属参数列表，优化器可以直接批量读取、批量迭代更新权重，是搭建网络层权重、偏置的标准写法。
具象运行示例：模型内部同时定义 self.w1=torch.tensor([1.0],requires_grad=True)（普通可训练张量）、self.w2=nn.Parameter(torch.tensor([1.0]))（参数类实例），遍历model.named_parameters()只会打印w2，w1全程不会出现。
> 补充关键疑问：为什么不直接用requires_grad=True来判定、自动收录？
> 答：requires_grad只负责控制【是否计算梯度】，不能等同于【这是模型权重，需要优化更新+持久保存】。
> 前向/反向计算里会产生大量临时张量，它们也可能临时开启requires_grad=True（只求梯度做计算，本身不是网络权重，绝对不能交给优化器更新、不需要存入权重文件）。
> 如果只靠requires_grad=True识别参数，框架会把临时中间变量误当成权重更新，引发错误。
> 所以PyTorch做了解耦：requires_grad管「求不求梯度」；nn.Parameter类型标记管「是不是模型权重、纳入参数管理」。

2. 梯度开关默认值差异
普通Tensor新建后默认requires_grad=False，不会追踪运算梯度，需要手动调用requires_grad_(True)开启；
nn.Parameter创建完成默认requires_grad=True，适配权重训练场景，想要冻结权重只需手动关闭该开关即可。
配套示例：直接新建普通张量梯度开关关闭，新建Parameter梯度开关默认打开。

3. 使用定位差异
普通Tensor适合存放输入样本、固定运算常量、训练过程产生的临时中间结果；
nn.Parameter专门用来存储网络可训练权重、偏置，天然适配模型保存、多设备迁移、迁移学习分层冻结权重等工程场景。

核心疑问预先解答：
1. model(输入张量)调用不会传参报错：nn.Module内置__call__魔术方法，编写model(输入)时，框架先执行训练状态校验、钩子触发等前置逻辑，再自动调用forward(传入参数)；self是类实例调用时隐传入的参数，外部只需填写业务输入即可。
2. super().__init__()不能省略：作用是初始化父类内置的参数存储容器，后续才能自动扫描收集类内全部nn.Parameter成员，供优化器批量读取；省略后自动收录功能失效。
3. requires_grad默认开启规则：新建Parameter默认追踪梯度，普通Tensor需要手动开启梯度追踪开关。
"""

# ====================== 第一部分：普通Tensor 和 nn.Parameter 基础对照示例 ======================
# 例1：梯度开关默认状态对比
normal_weight = torch.tensor([1.5])  # 新建普通张量，默认关闭梯度追踪，反向传播不会计算它的梯度
param_weight = nn.Parameter(torch.tensor([1.5]))  # 新建Parameter实例，默认开启梯度追踪，适配权重训练
print("=== 梯度开关默认状态对照 ===")
print(f"普通张量requires_grad：{normal_weight.requires_grad}")  # 输出False，默认不追踪梯度
print(f"Parameter默认requires_grad：{param_weight.requires_grad}\n")  # 输出True，默认开启梯度追踪

# 手动为普通张量开启梯度追踪，开启后可以正常参与反向求导
normal_weight.requires_grad_(True)

# ====================== 第二部分：搭建单层线性模型，验证自动注册差异 + forward调用逻辑 ======================
class SimpleLinear(nn.Module):
    def __init__(self):
        # 调用父类nn.Module初始化方法，创建内置参数存储容器，省略会导致Parameter自动收录失效，必不可少
        super().__init__()

        # 1.Parameter类型权重：被父类自动收录进参数列表，优化器遍历可读取，执行step时自动更新数值
        self.w_param = nn.Parameter(torch.tensor([1.5]))
        self.b_param = nn.Parameter(torch.tensor([0.2]))

        # 2.普通可训练张量：手动开启梯度追踪，能参与运算、反向传播可算出梯度，但不会被模型自动收录，优化器读取不到
        self.w_normal = torch.tensor([3.0], requires_grad=True)

        # 3.普通常量张量：不开启梯度追踪，仅参与数值运算，不会收录进参数列表、不计算梯度
        self.fixed_const = torch.tensor([2.0])

    def forward(self, inp):
        """
        前向计算函数：自定义运算逻辑，不推荐手动直接调用forward
        使用 model(输入) 调用模型实例，会触发内置__call__方法自动执行本函数、传入inp参数
        参数匹配说明：self属于实例自带隐式参数，外部只需传入业务输入inp，不会触发参数数量报错
        运算公式：输出 = Parameter权重*输入 + 普通张量*固定常量 + 偏置
        """
        out = self.w_param * inp + self.w_normal * self.fixed_const + self.b_param
        return out

# 实例化模型，类内所有Parameter成员已完成自动登记
model = SimpleLinear()

# 遍历模型自动收录的Parameter，验证普通张量不会被采集
print("=== 模型自动收录的可训练参数（仅Parameter类型） ===")
for param_name, param_val in model.named_parameters():
    print(f"参数名：{param_name}，初始数值：{param_val.item()}")
print("提示：w_normal、fixed_const不会出现在列表中，普通张量无法被优化器批量读取\n")

# ====================== 第三部分：完整训练流程演示（前向→求梯度→优化更新） ======================
input_data = torch.tensor([2.0])
# 调用模型实例触发__call__内部自动运行forward(input_data)，完成前向预测
pred_result = model(input_data)
print(f"=== 前向计算结果 ===")
print(f"输入值：{input_data.item()}，模型预测输出：{pred_result.item()}")

# 构造简单平方损失，反向传播会为所有开启梯度追踪的变量计算梯度（普通张量、Parameter均可算出梯度）
loss = pred_result ** 2
loss.backward()

print(f"\n=== 反向传播梯度结果对照 ===")
print(f"Parameter权重w_param梯度：{model.w_param.grad.item()}")
print(f"普通张量w_normal梯度：{model.w_normal.grad.item()}（两者都能正常计算梯度，区别不在求导能力）")

# 优化器仅读取模型自动收录的Parameter构建更新名单，普通张量w_normal不在名单内不会被更新
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
optimizer.step()  # 执行一步权重更新，仅修改Parameter数值
optimizer.zero_grad()  # 清空本轮梯度，避免下一轮迭代梯度累加出错

print(f"\n=== SGD更新后数值对照 ===")
print(f"Parameter权重w_param更新后：{model.w_param.item()}（被优化器自动更新）")
print(f"普通张量w_normal更新后：{model.w_normal.item()}（数值无变化，优化器采集不到不会处理）")

# ====================== 第四部分：参数冻结实操（固定权重不参与训练） ======================
model.w_param.requires_grad = False  # 手动冻结Parameter权重，停止梯度刷新、优化器不再更新该参数
new_pred = model(input_data)
new_loss = new_pred ** 2
new_loss.backward()
print(f"\n=== 冻结w_param再次反向 ===")
print(f"冻结状态w_param梯度：{model.w_param.grad}（冻结后不会刷新梯度值）")
print(f"偏置b_param正常计算梯度：{model.b_param.grad.item()}")


=== 梯度开关默认状态对照 ===
普通张量requires_grad：False
Parameter默认requires_grad：True

=== 模型自动收录的可训练参数（仅Parameter类型） ===
参数名：w_param，初始数值：1.5
参数名：b_param，初始数值：0.20000000298023224
提示：w_normal、fixed_const不会出现在列表中，普通张量无法被优化器批量读取

=== 前向计算结果 ===
输入值：2.0，模型预测输出：9.199999809265137

=== 反向传播梯度结果对照 ===
Parameter权重w_param梯度：36.79999923706055
普通张量w_normal梯度：36.79999923706055（两者都能正常计算梯度，区别不在求导能力）


/home/fzus/anaconda3/envs/cs336/lib/python3.10/site-packages/torch/autograd/graph.py:979: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass



=== SGD更新后数值对照 ===
Parameter权重w_param更新后：-2.180000066757202（被优化器自动更新）
普通张量w_normal更新后：3.0（数值无变化，优化器采集不到不会处理）

=== 冻结w_param再次反向 ===
冻结状态w_param梯度：None（冻结后不会刷新梯度值）
偏置b_param正常计算梯度：-2.384185791015625e-07
